# 05 COF Property Prediction: Build Your First Reliable Baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanteen/COF-ML-Tutorial/blob/main/notebooks/en/05_cof_property_prediction.ipynb)

This is the final chapter of the core route. We now connect data, features, targets, splitting and evaluation into one workflow.

## 1. What will we do?
We use a teaching dataset to predict a synthetic target called `CO2_uptake_demo`.

Workflow:
`choose features → split train/test → preprocess → train → predict → calculate errors → judge reliability`

**Important: this target is synthetic and must not be used for scientific conclusions.**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
url='https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/data/cof_demo.csv'
df=pd.read_csv(url)
display(df.head())
print('shape =',df.shape)

## 2. Choose inputs and output
Numerical features can be used directly. Text categories such as `family` and `functional_group` must be converted into numbers.

We use `OneHotEncoder` for this conversion. You do not need the mathematical details yet; the key idea is that the model ultimately receives numbers.

In [ ]:
target='CO2_uptake_demo'
features=['family','functional_group','pore_A','void_fraction','density','N_fraction','O_fraction']
X=df[features]; y=df[target]
cat=['family','functional_group']
num=[c for c in features if c not in cat]
pre=ColumnTransformer([
 ('num',StandardScaler(),num),
 ('cat',OneHotEncoder(handle_unknown='ignore'),cat)
])
model=Pipeline([
 ('pre',pre),
 ('rf',RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1))
])
print('features:',features)
print('target  :',target)

## 3. Start with a random split
A random split randomly assigns part of the samples to the test set. It is a useful first baseline, but for COFs it may not be strict enough.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
model.fit(X_train,y_train)
pred=model.predict(X_test)
print('MAE =',mean_absolute_error(y_test,pred))
print('RMSE=',mean_squared_error(y_test,pred)**0.5)
print('R2  =',r2_score(y_test,pred))

## 4. Why use a family-aware split for COFs?
Structures within the same COF family may be very similar. With a random split, members of one family can appear in both training and test sets. The model is then tested on structures that are close to what it has already seen.

If your scientific goal is to predict **unseen COF families**, keep entire families together when splitting. This is a **family-aware split**.

In [ ]:
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
train_idx,test_idx=next(splitter.split(X,y,groups=df['family']))
X_train2,X_test2=X.iloc[train_idx],X.iloc[test_idx]
y_train2,y_test2=y.iloc[train_idx],y.iloc[test_idx]
model.fit(X_train2,y_train2)
pred2=model.predict(X_test2)
print('Train families:',sorted(df.iloc[train_idx]['family'].unique()))
print('Test families :',sorted(df.iloc[test_idx]['family'].unique()))
print('MAE =',mean_absolute_error(y_test2,pred2))
print('RMSE=',mean_squared_error(y_test2,pred2)**0.5)
print('R2  =',r2_score(y_test2,pred2))

## 5. How should we interpret the two results?
If the random split performs well but the family-aware split is much worse, the code is not necessarily wrong. The model may interpolate well near familiar chemical space but extrapolate poorly to new families.

This leads to two important ideas: **generalization** and **applicability domain**.

## Glossary
- baseline: a simple reference model;
- preprocessing: data transformations before training;
- one-hot encoding: conversion of categories into numerical columns;
- random split: random train/test partition;
- group/family-aware split: keep groups intact during splitting;
- generalization: performance on unseen data;
- applicability domain: the region of materials space where a model is expected to be reliable.

## Exercises
1. Remove `functional_group` and retrain.
2. Remove `pore_A` and `void_fraction` and retrain.
3. Compare random and family-aware splitting.
4. Explain why the highest R² is not the only goal in materials ML.
5. Write one sentence describing model performance and explicitly state which split was used.

### Core-course completion standard
If you can explain `feature → split → train → predict → error → validation` and understand why COF-family leakage matters, you have completed the core 00–05 route.